In [ ]:
%pip install torch --force-reinstall --index-url https://download.pytorch.org/whl/cu121
%pip install -q peft
%pip install -q datasets
%pip install -q trl

In [1]:
import torch

print(f"Versión de PyTorch: {torch.__version__}")
print(f"¿CUDA disponible?: {torch.cuda.is_available()}")

if not torch.cuda.is_available():
    print(f"Dispositivos CUDA detectados por driver: {torch.cuda.device_count()}")

Versión de PyTorch: 2.15.0+cu132
¿CUDA disponible?: True


In [2]:
from huggingface_hub import login
import os
from dotenv import load_dotenv

token = os.getenv("TOKEN")

login(token)

In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_id = "meta-llama/Meta-Llama-3.1-8B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",               
    bnb_4bit_compute_dtype=torch.bfloat16, 
    bnb_4bit_use_double_quant=True           
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"                        # Asigna las capas automáticamente a la GPU
)

print("¡Modelo cargado correctamente en VRAM!")

c:\Users\mroscardd\Desktop\Code\IA Keepcoding\proyectos\BOE_Model\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1106 11:35:14.170000 10980 Lib\site-packages\torch\utils\flop_counter.py:113] triton not found; flop counting will not work for triton kernels
Loading weights: 100%|██████████| 291/291 [00:09<00:00, 30.92it/s]


¡Modelo cargado correctamente en VRAM!


In [4]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training


# 1. Habilitar gradientes y preparar el modelo de 4 bits para entrenamiento
model = prepare_model_for_kbit_training(model)

# 2. Configurar el adaptador LoRA
peft_config = LoraConfig(
    r=16,                           # Rango (rank) de las matrices de adaptación
    lora_alpha=32,                  # Factor de escalado de los gradientes
    target_modules=[                # Capas donde inyectamos los adaptadores
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# 3. Aplicar la configuración al modelo
model = get_peft_model(model, peft_config)

# 4. Imprimir parámetros entrenables vs totales
model.print_trainable_parameters()

trainable params: 41,943,040 || all params: 8,072,204,288 || trainable%: 0.5196


In [14]:
from trl import SFTConfig, SFTTrainer
from datasets import load_dataset


dataset_data = load_dataset(
    "json",
    data_files="./dataset_qlora/dataset_redaccion_juridica.json",
    split="train"
)


sft_config = SFTConfig(
    output_dir="./resultados_qlora",
    max_length=512,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    warmup_steps=2,
    max_steps=10,
    learning_rate=2e-4,
    fp16=False,
    bf16=True,
    logging_steps=1,
    optim="paged_adamw_8bit",
    save_strategy="no",
    loss_type="nll",
    max_grad_norm=0.0,
)


# 5. Inicializar SFTTrainer
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset_data,
    processing_class=tokenizer,
    args=sft_config,
)

trainer.train()


Generating train split: 105 examples [00:00, 7225.74 examples/s]
Truncating train dataset: 100%|██████████| 105/105 [00:00<00:00, 8783.09 examples/s]
Dropping fully masked examples from train dataset: 100%|██████████| 105/105 [00:00<00:00, 11668.44 examples/s]


Step,Training Loss
1,0.687766
2,0.664419
3,0.563626
4,0.536672
5,0.502529
6,0.466607
7,0.493496
8,0.460825
9,0.469015
10,0.498226


TrainOutput(global_step=10, training_loss=0.5343181073665619, metrics={'train_runtime': 17.4565, 'train_samples_per_second': 4.583, 'train_steps_per_second': 0.573, 'total_flos': 810171342618624.0, 'train_loss': 0.5343181073665619, 'epoch': 0.7547169811320755})

In [6]:
model.save_pretrained("./adaptador_qlora")
tokenizer.save_pretrained("./adaptador_qlora")

('./adaptador_qlora\\tokenizer_config.json',
 './adaptador_qlora\\chat_template.jinja',
 './adaptador_qlora\\tokenizer.json')

In [13]:
# modo inferencia
model.eval()

messages = [
    {
        "role": "system",
        "content": "Eres un asistente especializado en derecho español. Responde con rigor jurídico y no inventes referencias normativas."
    },
    {
        "role": "user",
        "content": "¿Tengo derecho a una vivienda digna?"
    }
]

prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(prompt, return_tensors="pt").to(model.get_input_embeddings().weight.device)

# Generar respuesta con parámetros de control
with torch.no_grad():
    outputs = model.generate(
    **inputs,
    max_new_tokens=100,
    do_sample=True,             
    temperature=0.7,
    top_p=0.9,
    repetition_penalty=1.2,
    pad_token_id=tokenizer.eos_token_id
)

    
input_length = inputs.input_ids.shape[1]
respuesta_limpia = tokenizer.decode(outputs[0][input_length:], skip_special_tokens=True)

# Decodificar e imprimir

print(respuesta_limpia)

El derecho a la vivienda es considerado como uno de los derechos sociales fundamentales del artículo 47 de la Constitución Española, junto al agua, electricidad o salud. Esta protección general tiene su desarrollo en el artículos 46.1.2º de la LOFFA (Ley Orgánica 8/2006), estableciendo que "toda persona tiene derecho a disfrutar de las condiciones de vida adecuadas para sí misma y su familia".


